# Supplementary Practical – PyTorch Fundamentals with ANN, CNN, and RNN

**Course:** ML6402 – Machine Learning  
**Instructor:** Dr. Ahmed Sabbah  
**University:** Al-Quds Open University  

This notebook is **additional knowledge**. The main deep-learning implementation in ML6402 uses TensorFlow/Keras, while this notebook introduces PyTorch for students who want a more explicit, research-oriented deep-learning workflow.

## Learning objectives

By the end of this notebook, you should be able to:

- create and manipulate PyTorch tensors;
- move tensors and models between CPU and GPU;
- use `TensorDataset` and `DataLoader`;
- define a neural network using `nn.Module`;
- understand the forward pass, loss function, backpropagation, and optimizer;
- write reusable training and evaluation loops;
- implement a simple ANN;
- implement a simple CNN;
- implement a simple RNN;
- make predictions;
- save and reload model parameters.

The examples are intentionally small so that the focus remains on the **PyTorch programming model** rather than on obtaining state-of-the-art performance.

## 1. Import PyTorch and check the execution device

PyTorch can run on the CPU or, when available, on a GPU. We choose the device once and then move both the model and its tensors to that device.

### Code explanation: imports and device selection

This cell prepares the libraries used throughout the notebook.

- `torch` is the main PyTorch package.
- `torch.nn` contains neural-network layers, activation functions, and loss functions.
- `TensorDataset` groups input tensors and target tensors into one dataset object.
- `DataLoader` creates mini-batches and optionally shuffles the training samples.
- NumPy and Pandas are used for ordinary data manipulation.
- Matplotlib is used for plots.
- Scikit-Learn provides the datasets, train/test splitting, standardization, and evaluation metrics.

The line:

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

checks whether an NVIDIA CUDA-capable GPU is available. If it is, PyTorch uses the GPU; otherwise, it uses the CPU.

**Important:** moving only the model to the GPU is not enough. The input tensors must also be moved to the same device.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

print("PyTorch version:", torch.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. PyTorch tensors

A PyTorch tensor is similar to a NumPy array, but it can also:

- run on a GPU;
- participate in automatic differentiation;
- be used directly by neural-network layers.

### Code explanation: creating and inspecting tensors

`torch.tensor(...)` converts Python values into a PyTorch tensor.

In this example:

```text
shape = [2, 2]
```

meaning that the tensor has 2 rows and 2 columns.

`x.dtype` shows the data type. Because the numbers were written with decimal points, PyTorch creates floating-point values.

The expression:

```python
x + 2
```

performs element-wise addition.

The expression:

```python
x @ x.T
```

performs matrix multiplication. `x.T` means the transpose of `x`.

This is important because neural networks perform many tensor and matrix operations internally.

In [ ]:
x = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])

print("Tensor:")
print(x)
print("Shape:", x.shape)
print("Data type:", x.dtype)

print("\nBasic operations:")
print("x + 2 =")
print(x + 2)

print("\nMatrix multiplication:")
print(x @ x.T)

### Automatic differentiation

When `requires_grad=True`, PyTorch records operations so that gradients can be computed automatically during backpropagation.

### Code explanation: automatic differentiation

This small example demonstrates one of PyTorch's most important features: **autograd**.

We define:

```python
w = torch.tensor(2.0, requires_grad=True)
```

The argument `requires_grad=True` tells PyTorch to track operations involving `w`.

The expression is:

```text
y = w × x²
```

With `w = 2` and `x = 3`:

```text
y = 2 × 3² = 18
```

Calling:

```python
y.backward()
```

asks PyTorch to compute the derivative of `y` with respect to every tracked parameter.

Here:

```text
dy/dw = x² = 9
```

and PyTorch stores this value in:

```python
w.grad
```

This same mechanism is used during neural-network training. Instead of manually calculating derivatives for thousands or millions of weights, PyTorch computes them automatically.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
x_value = torch.tensor(3.0)

y = w * x_value ** 2
y.backward()

print("y =", y.item())
print("dy/dw =", w.grad.item())

## 3. Dataset and DataLoader

A common PyTorch workflow is:

`NumPy/Pandas → Tensor → Dataset → DataLoader`

A `DataLoader` divides the data into mini-batches and can shuffle the training samples.

### Code explanation: Dataset and DataLoader

The tensors `X_demo` and `y_demo` represent four simple input-target pairs.

`TensorDataset(X_demo, y_demo)` connects each input with its corresponding target.

Then:

```python
DataLoader(..., batch_size=2, shuffle=True)
```

means:

- use only 2 samples in each mini-batch;
- randomly reorder the training samples before each pass through the dataset.

The `for` loop shows what the DataLoader returns.

This is the same pattern we will use later:

```text
Dataset → DataLoader → batches → model
```

Mini-batches are useful because they reduce memory requirements and allow the optimizer to update the model several times during one epoch.

In [ ]:
X_demo = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y_demo = torch.tensor([[2.0], [4.0], [6.0], [8.0]])

demo_dataset = TensorDataset(X_demo, y_demo)
demo_loader = DataLoader(demo_dataset, batch_size=2, shuffle=True)

for X_batch, y_batch in demo_loader:
    print("X batch:", X_batch.squeeze().tolist())
    print("y batch:", y_batch.squeeze().tolist())
    print("---")

## 4. A reusable PyTorch training loop

The central training steps are:

1. obtain a mini-batch;
2. move it to the selected device;
3. clear old gradients;
4. perform the forward pass;
5. compute the loss;
6. call `loss.backward()` to compute gradients;
7. call `optimizer.step()` to update the parameters.

### Code explanation: the classification training loop

This cell defines two reusable functions.

#### `train_classifier(...)`

`model.to(device)` moves the network parameters to CPU or GPU.

For every epoch:

```python
model.train()
```

puts the model into training mode.

Then, for every mini-batch:

1. `X_batch.to(device)` and `y_batch.to(device)` move data to the same device as the model.
2. `optimizer.zero_grad()` clears gradients left from the previous mini-batch.
3. `logits = model(X_batch)` performs the **forward pass**.
4. `loss = loss_fn(logits, y_batch)` measures prediction error.
5. `loss.backward()` performs **backpropagation** and calculates gradients.
6. `optimizer.step()` updates the model weights.

#### `evaluate_classifier(...)`

`model.eval()` switches the network to evaluation mode.

`torch.no_grad()` disables gradient tracking because no learning occurs during evaluation. This saves memory and computation.

`torch.argmax(logits, dim=1)` chooses the class with the largest output score.

Finally, predictions are copied back to CPU with `.cpu()` before conversion to NumPy.

In [ ]:
def train_classifier(model, loader, loss_fn, optimizer, epochs=20):
    model.to(device)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)

        if epoch == 0 or (epoch + 1) % 5 == 0:
            epoch_loss = running_loss / len(loader.dataset)
            print(f"Epoch {epoch + 1:>2}/{epochs} - loss: {epoch_loss:.4f}")


def evaluate_classifier(model, loader):
    model.eval()
    predictions = []
    targets = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            pred = torch.argmax(logits, dim=1)

            predictions.extend(pred.cpu().numpy())
            targets.extend(y_batch.numpy())

    return accuracy_score(targets, predictions)

# Part A – Artificial Neural Network (ANN)

We will classify the Scikit-Learn breast-cancer dataset.

The ANN demonstrates:

- fully connected `Linear` layers;
- ReLU activation;
- mini-batch training;
- `CrossEntropyLoss`;
- Adam optimizer.

### Code explanation: preparing ANN data

The breast-cancer dataset contains numerical measurements and a binary target.

We convert:

- features to `np.float32`, which is the common numeric type for neural-network inputs;
- class labels to `np.int64`, which is required by PyTorch's `CrossEntropyLoss`.

The train/test split uses:

```python
stratify=y
```

so the class proportions are approximately preserved in both sets.

The scaler is fitted **only on the training data**:

```python
scaler.fit_transform(X_train)
```

and then applied to the test data:

```python
scaler.transform(X_test)
```

This avoids data leakage.

Finally:

```python
torch.from_numpy(...)
```

converts NumPy arrays into tensors, and the DataLoaders create mini-batches.

Notice that the training DataLoader uses `shuffle=True`, but the test DataLoader does not need shuffling.

In [ ]:
data = load_breast_cancer()
X = data.data.astype(np.float32)
y = data.target.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train)
)
test_dataset = TensorDataset(
    torch.from_numpy(X_test),
    torch.from_numpy(y_test)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("Number of input features:", X_train.shape[1])
print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

## 5. Define the ANN using `nn.Module`

Every custom PyTorch network normally inherits from `nn.Module`.

- `__init__()` defines the layers.
- `forward()` defines how data moves through those layers.

### Code explanation: ANN architecture

The ANN is created by defining a Python class that inherits from:

```python
nn.Module
```

Inside `__init__()`, we define the layers.

The architecture is:

```text
input features
    ↓
Linear(input_dim → 32)
    ↓
ReLU
    ↓
Linear(32 → 16)
    ↓
ReLU
    ↓
Linear(16 → 2)
    ↓
two class logits
```

`nn.Linear(a, b)` means a fully connected layer with `a` input values and `b` output values.

`ReLU` introduces non-linearity:

```text
ReLU(z) = max(0, z)
```

Without non-linear activation functions, stacking many linear layers would still behave like one linear transformation.

The `forward()` method defines how one input tensor flows through the network.

In [ ]:
class ANNClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.network(x)


ann_model = ANNClassifier(input_dim=X_train.shape[1])
print(ann_model)

### Why does the final layer have 2 outputs?

There are two classes. The network produces two raw scores called **logits**.

When we use `CrossEntropyLoss`, we should **not** manually apply softmax inside the model. The loss function handles the required transformation internally.

### Code explanation: loss, optimizer, and ANN training

We use:

```python
nn.CrossEntropyLoss()
```

because this is a classification problem.

We use:

```python
torch.optim.Adam(...)
```

as the optimizer. Adam adjusts model parameters using the gradients produced by backpropagation.

The learning rate:

```python
lr=0.001
```

controls the size of each parameter update.

Then the generic training function is called for 20 epochs.

After training, `evaluate_classifier(...)` calculates accuracy on the test set.

Students should observe that the printed loss generally decreases during training. A decreasing loss means that the model is becoming better at fitting the training data, although test performance is still needed to judge generalization.

In [ ]:
ann_loss = nn.CrossEntropyLoss()
ann_optimizer = torch.optim.Adam(ann_model.parameters(), lr=0.001)

train_classifier(
    ann_model,
    train_loader,
    ann_loss,
    ann_optimizer,
    epochs=20
)

ann_accuracy = evaluate_classifier(ann_model, test_loader)
print("ANN test accuracy:", round(ann_accuracy, 4))

## 6. Make a prediction with the ANN

### Code explanation: ANN prediction

Before prediction:

```python
ann_model.eval()
```

puts the model into evaluation mode.

We select one standardized test sample and convert it to a tensor.

Inside `torch.no_grad()`:

```python
logits = ann_model(sample)
```

performs the forward pass.

The result contains one score for each class.

```python
torch.argmax(logits, dim=1)
```

returns the index of the largest score, which becomes the predicted class.

This illustrates an important point:

```text
network output → logits → choose largest logit → predicted class
```

In [ ]:
ann_model.eval()

sample = torch.from_numpy(X_test[:1]).to(device)

with torch.no_grad():
    logits = ann_model(sample)
    predicted_class = torch.argmax(logits, dim=1).item()

print("Predicted class:", data.target_names[predicted_class])
print("Actual class   :", data.target_names[y_test[0]])

# Part B – Convolutional Neural Network (CNN)

A CNN is designed to exploit spatial structure in images.

We use the Scikit-Learn handwritten-digits dataset:

- 8 × 8 grayscale images;
- 10 classes: digits 0–9.

The example demonstrates:

- adding an image-channel dimension;
- `Conv2d`;
- ReLU;
- max pooling;
- flattening;
- fully connected classification.

### Code explanation: preparing image data for the CNN

The digits dataset contains grayscale images of size:

```text
8 × 8
```

Each pixel originally ranges from 0 to 16. We divide by 16 so values fall approximately between 0 and 1.

A CNN in PyTorch expects image tensors in this order:

```text
[samples, channels, height, width]
```

The original dataset has shape:

```text
[samples, 8, 8]
```

Because grayscale images have one channel, we add a new dimension using:

```python
np.newaxis
```

The new shape becomes:

```text
[samples, 1, 8, 8]
```

This shape convention is essential when working with `nn.Conv2d`.

In [ ]:
digits = load_digits()

X_img = digits.images.astype(np.float32) / 16.0
y_img = digits.target.astype(np.int64)

X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(
    X_img, y_img,
    test_size=0.20,
    random_state=42,
    stratify=y_img
)

# CNN input shape: [samples, channels, height, width]
X_train_img = X_train_img[:, np.newaxis, :, :]
X_test_img = X_test_img[:, np.newaxis, :, :]

train_img_dataset = TensorDataset(
    torch.from_numpy(X_train_img),
    torch.from_numpy(y_train_img)
)
test_img_dataset = TensorDataset(
    torch.from_numpy(X_test_img),
    torch.from_numpy(y_test_img)
)

train_img_loader = DataLoader(train_img_dataset, batch_size=64, shuffle=True)
test_img_loader = DataLoader(test_img_dataset, batch_size=128, shuffle=False)

print("CNN training tensor shape:", X_train_img.shape)

## 7. Define the CNN

For an 8 × 8 image:

- first convolution preserves the spatial size;
- max pooling reduces 8 × 8 to 4 × 4;
- second convolution preserves 4 × 4;
- second max pooling reduces 4 × 4 to 2 × 2.

With 32 output channels, the flattened representation therefore contains:

`32 × 2 × 2 = 128` values.

### Code explanation: CNN architecture and tensor shapes

The CNN has two main parts.

#### Feature extractor

The first layer:

```python
nn.Conv2d(1, 16, kernel_size=3, padding=1)
```

means:

- 1 input channel;
- 16 learned filters;
- each filter is 3 × 3;
- padding keeps the image size unchanged.

So:

```text
1 × 8 × 8 → 16 × 8 × 8
```

Then max pooling reduces width and height:

```text
16 × 8 × 8 → 16 × 4 × 4
```

The second convolution produces 32 channels:

```text
16 × 4 × 4 → 32 × 4 × 4
```

and the second pooling gives:

```text
32 × 4 × 4 → 32 × 2 × 2
```

#### Classifier

`nn.Flatten()` converts:

```text
32 × 2 × 2
```

into:

```text
128 values
```

The dense layers then map:

```text
128 → 64 → 10
```

where 10 outputs represent the digits 0 through 9.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 2 * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


cnn_model = SimpleCNN()
print(cnn_model)

### Code explanation: CNN training

The CNN uses the same general training loop as the ANN.

That is one of the major advantages of PyTorch: after the model class is defined, the overall training process remains almost identical.

We again use:

- `CrossEntropyLoss` for multi-class classification;
- Adam optimizer;
- learning rate 0.001.

The important difference is not the training loop—it is the **network architecture** and the **shape of the input data**.

Students should compare this with the ANN and notice:

```text
ANN: vector input → Linear layers
CNN: image tensor → Conv/Pool layers → Linear layers
```

In [ ]:
cnn_loss = nn.CrossEntropyLoss()
cnn_optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)

train_classifier(
    cnn_model,
    train_img_loader,
    cnn_loss,
    cnn_optimizer,
    epochs=10
)

cnn_accuracy = evaluate_classifier(cnn_model, test_img_loader)
print("CNN test accuracy:", round(cnn_accuracy, 4))

## 8. Inspect CNN predictions

### Code explanation: visualizing CNN predictions

We select eight test images and move them to the selected device.

The CNN predicts one class for each image.

Because plotting libraries operate on CPU/NumPy data, predictions are moved back using:

```python
.cpu().numpy()
```

The plot title uses:

- `P` = predicted digit;
- `A` = actual digit.

This visual inspection is useful because a numeric accuracy value alone does not show what kinds of mistakes the model makes.

In [ ]:
cnn_model.eval()

images = torch.from_numpy(X_test_img[:8]).to(device)

with torch.no_grad():
    predictions = torch.argmax(cnn_model(images), dim=1).cpu().numpy()

fig, axes = plt.subplots(2, 4, figsize=(8, 4))

for ax, image, pred, actual in zip(
    axes.ravel(),
    X_test_img[:8, 0],
    predictions,
    y_test_img[:8]
):
    ax.imshow(image, cmap="gray")
    ax.set_title(f"P:{pred} / A:{actual}")
    ax.axis("off")

plt.tight_layout()
plt.show()

# Part C – Recurrent Neural Network (RNN)

RNNs process sequences rather than independent feature vectors.

To keep the example self-contained, we create a noisy sine-wave time series and ask the RNN to predict the next value from a fixed-length window of previous values.

This demonstrates:

- sequential input shape;
- `nn.RNN`;
- hidden states;
- selecting the final time step;
- regression with `MSELoss`.

### Code explanation: creating a simple sequence

For the RNN example, we generate a noisy sine wave.

The clean part:

```python
np.sin(t)
```

creates a repeating temporal pattern.

The term:

```python
0.10 * np.random.randn(...)
```

adds random noise.

This creates a realistic teaching example where the next value depends strongly on previous values.

Unlike the ANN and CNN examples, the order of the data is important here. We must preserve temporal order when constructing training and test sequences.

In [ ]:
np.random.seed(42)

t = np.linspace(0, 50, 1200)
series = np.sin(t) + 0.10 * np.random.randn(len(t))

plt.plot(t[:300], series[:300])
plt.xlabel("Time")
plt.ylabel("Value")
plt.title("Synthetic time series")
plt.show()

## 9. Convert the time series into supervised sequences

If the sequence length is 20, each training example contains 20 consecutive values, and the target is the next value.

### Code explanation: sliding windows for RNN input

An RNN cannot directly learn from one long one-dimensional array. We convert the time series into many supervised examples.

With:

```text
SEQUENCE_LENGTH = 20
```

one sample looks conceptually like:

```text
Input:  [x1, x2, ..., x20]
Target: x21
```

The next sample becomes:

```text
Input:  [x2, x3, ..., x21]
Target: x22
```

After adding the final feature dimension, the RNN input has shape:

```text
[samples, time steps, features]
```

Here:

- time steps = 20;
- features = 1.

Unlike ordinary classification, we split the sequence by time rather than randomly shuffling train/test samples. This prevents future information from leaking into the training period.

In [ ]:
SEQUENCE_LENGTH = 20

def make_sequences(values, sequence_length):
    X_seq = []
    y_seq = []

    for i in range(len(values) - sequence_length):
        X_seq.append(values[i:i + sequence_length])
        y_seq.append(values[i + sequence_length])

    X_seq = np.array(X_seq, dtype=np.float32)[..., np.newaxis]
    y_seq = np.array(y_seq, dtype=np.float32).reshape(-1, 1)

    return X_seq, y_seq


X_seq, y_seq = make_sequences(series, SEQUENCE_LENGTH)

split = int(0.80 * len(X_seq))

X_train_seq = X_seq[:split]
X_test_seq = X_seq[split:]
y_train_seq = y_seq[:split]
y_test_seq = y_seq[split:]

train_seq_dataset = TensorDataset(
    torch.from_numpy(X_train_seq),
    torch.from_numpy(y_train_seq)
)
test_seq_dataset = TensorDataset(
    torch.from_numpy(X_test_seq),
    torch.from_numpy(y_test_seq)
)

train_seq_loader = DataLoader(train_seq_dataset, batch_size=32, shuffle=True)
test_seq_loader = DataLoader(test_seq_dataset, batch_size=64, shuffle=False)

print("RNN input shape:", X_train_seq.shape)
print("[samples, time steps, features]")

## 10. Define the RNN

With `batch_first=True`, the RNN expects:

`[batch, time steps, features]`

The RNN returns outputs for every time step. For next-value prediction, we use the output from the **last time step**.

### Code explanation: RNN architecture

The recurrent layer is:

```python
nn.RNN(input_size=1, hidden_size=32, batch_first=True)
```

- `input_size=1`: each time step contains one value.
- `hidden_size=32`: the RNN stores a 32-dimensional hidden representation.
- `batch_first=True`: inputs follow the shape `[batch, time, features]`.

The RNN returns:

```python
rnn_output, hidden_state
```

`rnn_output` contains an output representation for every time step.

For next-value forecasting, we use:

```python
rnn_output[:, -1, :]
```

which selects the final time step for every sample in the batch.

That representation is passed to:

```python
nn.Linear(hidden_size, 1)
```

to produce one predicted numeric value.

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=32):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x):
        rnn_output, hidden_state = self.rnn(x)

        last_time_step = rnn_output[:, -1, :]
        prediction = self.output(last_time_step)

        return prediction


rnn_model = SimpleRNN()
print(rnn_model)

## 11. Regression training loop for the RNN

The structure is almost identical to classification training. The main difference is the loss function and the form of the network output.

### Code explanation: RNN regression training

The regression loop follows the same logic as classification:

```text
forward pass → loss → backward pass → optimizer update
```

The important difference is the loss function.

For classification we used:

```python
CrossEntropyLoss
```

For numeric prediction we use:

```python
MSELoss
```

which computes mean squared error between predicted and actual values.

Again:

- `optimizer.zero_grad()` clears previous gradients;
- `loss.backward()` calculates new gradients;
- `optimizer.step()` updates the model.

This demonstrates that the PyTorch training framework is highly reusable across different neural-network types.

In [ ]:
def train_regressor(model, loader, loss_fn, optimizer, epochs=20):
    model.to(device)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            prediction = model(X_batch)
            loss = loss_fn(prediction, y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)

        if epoch == 0 or (epoch + 1) % 5 == 0:
            epoch_loss = running_loss / len(loader.dataset)
            print(f"Epoch {epoch + 1:>2}/{epochs} - MSE: {epoch_loss:.5f}")


rnn_loss = nn.MSELoss()
rnn_optimizer = torch.optim.Adam(rnn_model.parameters(), lr=0.001)

train_regressor(
    rnn_model,
    train_seq_loader,
    rnn_loss,
    rnn_optimizer,
    epochs=20
)

## 12. Evaluate the RNN

### Code explanation: evaluating the RNN

During evaluation:

```python
rnn_model.eval()
```

switches to evaluation mode, and:

```python
torch.no_grad()
```

disables gradient calculation.

Predictions from all test mini-batches are collected.

Then MSE is calculated manually:

```python
np.mean((prediction - actual) ** 2)
```

Finally, actual and predicted values are plotted together.

A useful RNN model should approximately follow the temporal pattern of the true sequence. The plot helps students see behavior that a single MSE number cannot fully communicate.

In [ ]:
rnn_model.eval()

rnn_predictions = []

with torch.no_grad():
    for X_batch, _ in test_seq_loader:
        X_batch = X_batch.to(device)
        pred = rnn_model(X_batch)
        rnn_predictions.extend(pred.cpu().numpy().ravel())

rnn_predictions = np.array(rnn_predictions)

mse = np.mean((rnn_predictions - y_test_seq.ravel()) ** 2)
print("RNN test MSE:", round(float(mse), 5))

plt.figure(figsize=(10, 4))
plt.plot(y_test_seq[:150], label="Actual")
plt.plot(rnn_predictions[:150], label="Predicted")
plt.xlabel("Test sequence")
plt.ylabel("Value")
plt.legend()
plt.show()

## RNN versus LSTM

`nn.RNN` is useful for understanding the recurrent idea, but plain RNNs can have difficulty learning long-term dependencies.

PyTorch also provides:

- `nn.LSTM`
- `nn.GRU`

For example, replacing:

```python
self.rnn = nn.RNN(...)
```

with:

```python
self.rnn = nn.LSTM(...)
```

creates an LSTM-based recurrent layer. The overall training workflow remains very similar.

## 13. Save and reload model parameters

The recommended PyTorch pattern is to save the model's **state dictionary**, not the whole Python object.

### Code explanation: saving and loading PyTorch models

`model.state_dict()` contains the learned parameters of the model, such as weights and biases.

We save them using:

```python
torch.save(model.state_dict(), "file.pth")
```

To restore them:

1. recreate the same model architecture;
2. load the saved state dictionary;
3. move the model to the desired device;
4. call `eval()` before inference.

The argument:

```python
map_location=device
```

allows a model saved on one device to be loaded on another—for example, a GPU-trained model can later be loaded on a CPU.

This pattern is more portable and is the standard recommended approach for saving PyTorch model parameters.

In [ ]:
# Save ANN parameters
torch.save(ann_model.state_dict(), "ann_model_weights.pth")

# Create a new model with the same architecture
reloaded_ann = ANNClassifier(input_dim=X_train.shape[1])

# Load parameters
reloaded_ann.load_state_dict(
    torch.load("ann_model_weights.pth", map_location=device)
)
reloaded_ann.to(device)
reloaded_ann.eval()

print("Model weights reloaded successfully.")

## 14. PyTorch workflow summary

The typical PyTorch workflow is:

1. **Prepare data**
   - NumPy / Pandas
   - convert to tensors
   - create Dataset
   - create DataLoader

2. **Define model**
   - subclass `nn.Module`
   - define layers in `__init__()`
   - define computation in `forward()`

3. **Choose training components**
   - loss function
   - optimizer
   - learning rate

4. **Train**
   - `model.train()`
   - zero gradients
   - forward pass
   - compute loss
   - `loss.backward()`
   - `optimizer.step()`

5. **Evaluate**
   - `model.eval()`
   - `torch.no_grad()`
   - calculate predictions and metrics

6. **Save**
   - `torch.save(model.state_dict(), ...)`

These steps are common across ANN, CNN, RNN, LSTM, Transformers, and many other PyTorch models.

## 15. Student exercises

### ANN
1. Change the hidden layers from `32 → 16` to `64 → 32 → 16`.
2. Compare Adam with SGD.
3. Try different learning rates.

### CNN
1. Remove one convolutional layer and compare accuracy.
2. Change the number of filters.
3. Explain why the input shape contains a channel dimension.

### RNN
1. Change the sequence length from 20 to 10 and 40.
2. Replace `nn.RNN` with `nn.LSTM`.
3. Compare the prediction curve and MSE.

### General PyTorch
1. Identify where the forward pass occurs.
2. Identify where gradients are calculated.
3. Explain the roles of `model.train()` and `model.eval()`.
4. Explain why `optimizer.zero_grad()` is called before backpropagation.
5. Explain why evaluation code uses `torch.no_grad()`.